## Зависимости

In [3]:
# !pip install transformers torch tqdm scikit-learn
# Раскомментировать при первом запуске

## Загрузка модели

`blanchefort/rubert-base-cased-sentiment` — предобученный RuBERT для классификации тональности русского текста. Три класса: POSITIVE / NEUTRAL / NEGATIVE.

In [ ]:
from transformers import pipeline

pipe = pipeline(
    'text-classification',
    model='blanchefort/rubert-base-cased-sentiment',
    device=-1,          # CPU; поменять на 0 при наличии GPU
    truncation=True,
    max_length=512,
    top_k=None,         # возвращаем вероятности всех 3 классов
)
print('Модель загружена')

## Данные

In [5]:
import pandas as pd
from pathlib import Path

news = pd.read_parquet('news_descriptions/news_collection_old.parquet')
news['date'] = pd.to_datetime(news['date']).dt.normalize()
print('Новостей:', len(news))
news.head(3)

Новостей: 79705


,Unnamed: 0,date,title,body,source
0,0,2024-09-09,no title,"""После падения добычи в 2022-2023 годах Газпро...",rdv
1,1,2024-09-09,no title,""" Рейтинг акций комьюнити РДВ. #опрос """,rdv
2,2,2024-09-09,no title,"""Нефть Brent упала до $71, Urals - до $60 за б...",rdv


## Предобработка текстов

In [6]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.strip().strip('"').strip()
    return text if len(text) >= 10 else ''

news['text_clean'] = news['body'].apply(clean_text)

empty = (news['text_clean'] == '').sum()
print(f'Пустых текстов: {empty} ({empty/len(news):.1%})')

Пустых текстов: 279 (0.4%)


## Инференс RuBERT

Если `rubert_scores.parquet` уже существует — инференс пропускается.

In [ ]:
from tqdm.auto import tqdm
import numpy as np

SCORES_PATH = Path('rubert_scores.parquet')
BATCH_SIZE  = 32

def to_score(probs):
    """P(POSITIVE) − P(NEGATIVE) → непрерывный скор от −1 до +1"""
    d = {p['label']: p['score'] for p in probs}
    return d.get('POSITIVE', 0) - d.get('NEGATIVE', 0)

def to_label(probs):
    return max(probs, key=lambda p: p['score'])['label']

if SCORES_PATH.exists():
    scores_df = pd.read_parquet(SCORES_PATH)
    print(f'Загружено из кэша: {len(scores_df)} записей')
else:
    texts = news['text_clean'].tolist()
    labels, raw_scores = [], []

    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='RuBERT'):
        batch = [t if t else '.' for t in texts[i : i + BATCH_SIZE]]
        for probs in pipe(batch):
            labels.append(to_label(probs))
            raw_scores.append(to_score(probs))

    scores_df = pd.DataFrame({
        'rubert_label': labels,
        'rubert_score': raw_scores,
    }, index=news.index)

    scores_df.to_parquet(SCORES_PATH)
    print(f'Сохранено: {SCORES_PATH}')

scores_df.head()

## Распределение классов

In [ ]:
import matplotlib.pyplot as plt

dist = scores_df['rubert_label'].value_counts()
print(dist)
print()
print(f'Доля негативных: {dist.get("NEGATIVE",0)/len(scores_df):.1%}')
print(f'Доля нейтральных: {dist.get("NEUTRAL",0)/len(scores_df):.1%}')
print(f'Доля позитивных: {dist.get("POSITIVE",0)/len(scores_df):.1%}')

fig, ax = plt.subplots(figsize=(5, 4))
dist.plot(kind='bar', ax=ax, color=['tomato','lightgrey','steelblue'], rot=0)
ax.set_title('Распределение тональности (RuBERT)')
ax.set_ylabel('Кол-во новостей')
plt.tight_layout()
plt.show()

## Сравнение с GPT-4o

Первые 15 000 новостей имеют разметку GPT-4o. Сравниваем два источника сентимента.

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
from scipy import stats

gpt4o = pd.read_json('news_descriptions/news_descriptions_GPT4o.json').transpose()
gpt4o['gpt_score'] = pd.to_numeric(gpt4o['sentiment_score'], errors='coerce')

def gpt_to_label(s):
    if s > 0.1:  return 'POSITIVE'
    if s < -0.1: return 'NEGATIVE'
    return 'NEUTRAL'

gpt4o['gpt_label'] = gpt4o['gpt_score'].apply(gpt_to_label)

compare = scores_df.iloc[:15000].join(gpt4o[['gpt_score','gpt_label']].reset_index(drop=True))
compare = compare.dropna()
print('Записей для сравнения:', len(compare))

r, p = stats.pearsonr(compare['rubert_score'], compare['gpt_score'])
kappa = cohen_kappa_score(compare['rubert_label'], compare['gpt_label'])
print(f'Pearson r    = {r:.3f}  (p={p:.4f})')
print(f"Cohen's κ    = {kappa:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(compare['gpt_score'], compare['rubert_score'], alpha=0.15, s=10)
axes[0].set_xlabel('GPT-4o sentiment')
axes[0].set_ylabel('RuBERT sentiment')
axes[0].set_title(f'Согласованность разметок  (r={r:.3f})')

labels_order = ['NEGATIVE','NEUTRAL','POSITIVE']
cm = confusion_matrix(compare['gpt_label'], compare['rubert_label'], labels=labels_order)
ConfusionMatrixDisplay(cm, display_labels=labels_order).plot(ax=axes[1], colorbar=False)
axes[1].set_title('GPT-4o (строки) vs RuBERT (столбцы)')

plt.tight_layout()
plt.show()

## Дневной индекс сентимента (RuBERT)

In [ ]:
news_with_scores = news.join(scores_df)

daily_rubert = (
    news_with_scores
    .groupby('date')['rubert_score']
    .agg(sentiment_index='mean', sent_std='std', news_count='count')
    .reset_index()
    .fillna({'sent_std': 0})
)
print('Дней с новостями:', len(daily_rubert))
daily_rubert.head()

## Корреляция RuBERT-сентимента с MOEX

In [ ]:
moex = pd.read_csv('MOEX.csv', encoding='cp1251', sep=';', engine='python', on_bad_lines='warn')
moex['date']   = pd.to_datetime(moex['TRADEDATE'], dayfirst=True)
moex['close']  = moex['CLOSE'].astype(str).str.replace(',', '.').astype(float)
moex['return'] = moex['close'].pct_change()
moex = moex[['date','close','return']].dropna().reset_index(drop=True)

final_rubert = moex.merge(daily_rubert, on='date', how='inner').reset_index(drop=True)
print('final_rubert:', final_rubert.shape)

r_rb, p_rb = stats.pearsonr(final_rubert['sentiment_index'], final_rubert['return'])
print(f'RuBERT  Pearson r = {r_rb:.3f}  p = {p_rb:.4f}')

## Визуализация: IMOEX и индекс сентимента RuBERT

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(final_rubert['date'], final_rubert['close'], color='steelblue', linewidth=1.2)
ax1.set_ylabel('IMOEX', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.fill_between(final_rubert['date'], final_rubert['sentiment_index'], alpha=0.25, color='tomato')
ax2.plot(final_rubert['date'], final_rubert['sentiment_index'], color='tomato', linewidth=0.8)
ax2.axhline(0, color='tomato', linewidth=0.5, linestyle='--')
ax2.set_ylabel('Индекс сентимента', color='tomato')
ax2.tick_params(axis='y', labelcolor='tomato')

ax1.set_title('Индекс МосБиржи и сентимент новостей (RuBERT)')
fig.legend(['IMOEX', 'Сентимент'], loc='upper left', bbox_to_anchor=(0.08, 0.92))
plt.tight_layout()
plt.show()